# Lab 04 — Spark SQL dan Data Analytics
**SQL/DataFrame, built-ins, aggregation, window, KPI, execution plan**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Menggunakan SQL dan DataFrame API sebagai dua pintu ke engine yang sama; membangun agregasi/KPI/window analysis; membaca execution plan dan menghubungkannya ke pertanyaan bisnis.

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 08:50:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.7
Master        : spark://spark-master:7077
App name      : spark-training-lab
Default parallelism: 2


## Dataset analitik sintetis
Dataset dibuat dengan Spark agar cukup besar untuk melihat plan/shuffle tetapi tetap aman untuk laptop lab.

In [2]:
from datetime import date

N = 50_000
transactions = (
    spark.range(N)
    .withColumn("trx_id", F.concat(F.lit("T"), F.lpad(F.col("id").cast("string"), 7, "0")))
    .withColumn("trx_date", F.date_add(F.lit("2026-08-01").cast("date"), (F.col("id") % 92).cast("int")))
    .withColumn("office", F.element_at(F.array(*[F.lit(x) for x in ["Jakarta Selatan", "Bandung", "Surabaya", "Bogor"]]), (F.col("id") % 4 + 1).cast("int")))
    .withColumn("tax_type", F.element_at(F.array(F.lit("PPN"), F.lit("PPh"), F.lit("PBB")), (F.col("id") % 3 + 1).cast("int")))
    .withColumn("channel", F.element_at(F.array(F.lit("Portal"), F.lit("API"), F.lit("Manual")), (F.col("id") % 3 + 1).cast("int")))
    .withColumn("status", F.when((F.col("id") % 11) == 0, F.lit("CANCEL")).otherwise(F.lit("VALID")))
    .withColumn("amount", ((F.col("id") * 7919) % 20_000_000 + 50_000).cast("double"))
    .drop("id")
)
transactions.createOrReplaceTempView("transactions")
transactions.show(5, truncate=False)
transactions.printSchema()
print("rows:", transactions.count())

+--------+----------+---------------+--------+-------+------+-------+
|trx_id  |trx_date  |office         |tax_type|channel|status|amount |
+--------+----------+---------------+--------+-------+------+-------+
|T0000000|2026-08-01|Jakarta Selatan|PPN     |Portal |CANCEL|50000.0|
|T0000001|2026-08-02|Bandung        |PPh     |API    |VALID |57919.0|
|T0000002|2026-08-03|Surabaya       |PBB     |Manual |VALID |65838.0|
|T0000003|2026-08-04|Bogor          |PPN     |Portal |VALID |73757.0|
|T0000004|2026-08-05|Jakarta Selatan|PPh     |API    |VALID |81676.0|
+--------+----------+---------------+--------+-------+------+-------+
only showing top 5 rows

root
 |-- trx_id: string (nullable = false)
 |-- trx_date: date (nullable = true)
 |-- office: string (nullable = true)
 |-- tax_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = false)
 |-- amount: double (nullable = true)



[Stage 1:=============================>                             (2 + 2) / 4]

rows: 50000


## Lab 1 — SQL vs DataFrame API (20 menit)
Tulis analisis yang sama dengan SQL dan DataFrame: transaksi VALID ≥ 1 juta, pilih office+amount, urutkan amount menurun, ambil 20.

In [3]:
sql_result = spark.sql("""
SELECT office, amount
FROM transactions
WHERE status = 'VALID' AND amount >= 1000000
ORDER BY amount DESC
LIMIT 20
""")

api_result = (
    transactions
    .filter((F.col("status") == "VALID") & (F.col("amount") >= 1_000_000))
    .select("office", "amount")
    .orderBy(F.col("amount").desc())
    .limit(20)
)

sql_result.show(5)
api_result.show(5)

+---------------+-----------+
|         office|     amount|
+---------------+-----------+
|        Bandung|2.0048871E7|
|       Surabaya| 2.004887E7|
|          Bogor|2.0048869E7|
|Jakarta Selatan| 2.004774E7|
|        Bandung|2.0047739E7|
+---------------+-----------+
only showing top 5 rows

+---------------+-----------+
|         office|     amount|
+---------------+-----------+
|        Bandung|2.0048871E7|
|       Surabaya| 2.004887E7|
|          Bogor|2.0048869E7|
|Jakarta Selatan| 2.004774E7|
|        Bandung|2.0047739E7|
+---------------+-----------+
only showing top 5 rows



In [ ]:
print("--- SQL PLAN ---")
sql_result.explain("formatted")
print("--- DATAFRAME PLAN ---")
api_result.explain("formatted")

**Diskusi:** apakah SQL otomatis lebih cepat daripada DataFrame API? Bandingkan physical plan, bukan selera sintaks.

## Lab 2 — Built-in functions, NULL, date, CASE (25 menit)

In [4]:
# Tambahkan beberapa NULL agar perilakunya terlihat.
with_null = transactions.withColumn(
    "amount2", F.when((F.xxhash64("trx_id") % 17) == 0, F.lit(None).cast("double")).otherwise(F.col("amount"))
)
with_null.createOrReplaceTempView("transactions_null")

spark.sql("""
SELECT
  COUNT(*) AS all_rows,
  COUNT(amount2) AS amount_not_null,
  SUM(CASE WHEN amount2 IS NULL THEN 1 ELSE 0 END) AS null_amount
FROM transactions_null
""").show()

+--------+---------------+-----------+
|all_rows|amount_not_null|null_amount|
+--------+---------------+-----------+
|   50000|          47074|       2926|
+--------+---------------+-----------+



In [7]:
spark.sql("""
SELECT
  trx_id,
  trx_date,
  YEAR(trx_date) AS yr,
  MONTH(trx_date) AS mn,
  office,
  amount,
  CASE
    WHEN amount >= 10000000 THEN 'HIGH'
    WHEN amount >= 1000000 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS amount_segment
FROM transactions
LIMIT 10
""").show(truncate=False)

+--------+----------+----+---+---------------+--------+--------------+
|trx_id  |trx_date  |yr  |mn |office         |amount  |amount_segment|
+--------+----------+----+---+---------------+--------+--------------+
|T0000000|2026-08-01|2026|8  |Jakarta Selatan|50000.0 |LOW           |
|T0000001|2026-08-02|2026|8  |Bandung        |57919.0 |LOW           |
|T0000002|2026-08-03|2026|8  |Surabaya       |65838.0 |LOW           |
|T0000003|2026-08-04|2026|8  |Bogor          |73757.0 |LOW           |
|T0000004|2026-08-05|2026|8  |Jakarta Selatan|81676.0 |LOW           |
|T0000005|2026-08-06|2026|8  |Bandung        |89595.0 |LOW           |
|T0000006|2026-08-07|2026|8  |Surabaya       |97514.0 |LOW           |
|T0000007|2026-08-08|2026|8  |Bogor          |105433.0|LOW           |
|T0000008|2026-08-09|2026|8  |Jakarta Selatan|113352.0|LOW           |
|T0000009|2026-08-10|2026|8  |Bandung        |121271.0|LOW           |
+--------+----------+----+---+---------------+--------+--------------+



In [8]:
spark.sql("""
SELECT 
office, 
tax_type, 
COUNT(*) AS trx_count, 
SUM(amount) AS total_amount,
AVG(amount) AS avg_amount
FROM transactions
WHERE status = 'VALID'
GROUP BY office, tax_type
""").show(truncate=False)

+---------------+--------+---------+---------------+-----------------+
|office         |tax_type|trx_count|total_amount   |avg_amount       |
+---------------+--------+---------+---------------+-----------------+
|Surabaya       |PPh     |3787     |3.775071403E10 |9968501.196197517|
|Bandung        |PPh     |3789     |3.7748749435E10|9962720.885457905|
|Surabaya       |PBB     |3789     |3.777467511E10 |9969563.238321457|
|Bandung        |PPN     |3787     |3.7744709165E10|9966915.5439662  |
|Bandung        |PBB     |3788     |3.7776665948E10|9972720.683210136|
|Surabaya       |PPN     |3788     |3.7742678732E10|9963748.34530095 |
|Jakarta Selatan|PPN     |3788     |3.774277376E10 |9963773.43189018 |
|Jakarta Selatan|PBB     |3787     |3.77587043E10  |9970611.116979139|
|Bogor          |PPh     |3788     |3.7768691516E10|9970615.500527984|
|Jakarta Selatan|PPh     |3788     |3.7790653164E10|9976413.190073919|
|Bogor          |PBB     |3787     |3.7736718895E10|9964805.623184578|
|Bogor

## Lab 3 — Aggregation + Window: ranking dan growth (35–40 menit)

In [ ]:
from pyspark.sql.window import Window

rank_w = Window.partitionBy("office").orderBy(F.col("amount").desc())
ranked = (
    transactions.filter(F.col("status") == "VALID")
    .withColumn("row_number", F.row_number().over(rank_w))
    .withColumn("rank", F.rank().over(rank_w))
    .withColumn("dense_rank", F.dense_rank().over(rank_w))
)
ranked.filter(F.col("row_number") <= 3).orderBy("office", "row_number").show(20, truncate=False)

In [ ]:
monthly = (
    transactions.filter(F.col("status") == "VALID")
    .withColumn("month", F.date_trunc("month", "trx_date"))
    .groupBy("office", "month")
    .agg(F.sum("amount").alias("total_amount"))
)

w = Window.partitionBy("office").orderBy("month")
monthly_growth = (
    monthly
    .withColumn("prev_amount", F.lag("total_amount").over(w))
    .withColumn("delta", F.col("total_amount") - F.col("prev_amount"))
    .withColumn("growth_pct", F.when(F.col("prev_amount") != 0, F.col("delta") / F.col("prev_amount") * 100))
)
monthly_growth.orderBy("office", "month").show(30, truncate=False)

## Lab 4 — Studi kasus analitik bisnis (40–50 menit)
Jawab tiga pertanyaan dan tulis insight dalam bahasa bisnis, bukan hanya tabel.

In [ ]:
# Q1: office terbesar + count + avg
q1 = (
    transactions.filter(F.col("status") == "VALID")
    .groupBy("office")
    .agg(
        F.count("*").alias("trx_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount")
    )
    .orderBy(F.col("total_amount").desc())
)
q1.show(truncate=False)

In [ ]:
# Q2: kontribusi channel per office
channel = (
    transactions.filter(F.col("status") == "VALID")
    .groupBy("office", "channel")
    .agg(F.sum("amount").alias("channel_amount"))
)
wo = Window.partitionBy("office")
wr = Window.partitionBy("office").orderBy(F.col("channel_amount").desc())
channel_share = (
    channel
    .withColumn("office_amount", F.sum("channel_amount").over(wo))
    .withColumn("share", F.col("channel_amount") / F.col("office_amount"))
    .withColumn("channel_rank", F.dense_rank().over(wr))
)
channel_share.orderBy("office", "channel_rank").show(30, truncate=False)

In [ ]:
# Q3: monthly trend + moving average 3 periode
w3 = Window.partitionBy("office").orderBy("month").rowsBetween(-2, 0)
trend = monthly_growth.withColumn("moving_avg_3", F.avg("total_amount").over(w3))
trend.orderBy("office", "month").show(30, truncate=False)

**Deliverable:** tulis 3 insight dan 1 keterbatasan analisis. Jangan menyimpulkan kausalitas dari pola agregat.

## Lab 5 — Membaca plan & menemukan bottleneck (30 menit)

In [9]:
q = (
    transactions
    .filter(F.col("status") == "VALID")
    .groupBy("office", "tax_type")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy(F.col("total_amount").desc())
)
q.explain("formatted")
q.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 50000, step=1, splits=Some(4))

(2) Filter
Input [1]: [id#0L]
Condition : NOT (((id#0L % 11) = 0) <=> true)

(3) Project
Output [3]: [element_at([Jakarta Selatan,Bandung,Surabaya,Bogor], cast(((id#0L % 4) + 1) as int), None, false) AS office#9, element_at([PPN,PPh,PBB], cast(((id#0L % 3) + 1) as int), None, false) AS tax_type#14, cast((((id#0L * 7919) % 20000000) + 50000) as double) AS amount#35]
Input [1]: [id#0L]

(4) HashAggregate
Input [3]: [office#9, tax_type#14, amount#35]
Keys [2]: [office#9, tax_type#14]
Functions [1]: [partial_sum(amount#35)]
Aggregate Attributes [1]: [sum#293]
Results [3]: [office#9, tax_type#14, sum#294]

(5) Exchange
Input [3]: [office#9, tax_type#14, sum#

**Checklist:**
1. Berapa Scan?
2. Di mana `Exchange` muncul dan operator apa yang memicunya?
3. Cocokkan SQL tab dengan Stages di Spark UI.
4. Apakah semua Exchange adalah masalah? Jelaskan.

## Mini Project — Dari data ke insight (30–40 menit)
Pilih satu:
- **Performance:** office mana memiliki volume/nilai tertinggi dan bagaimana tren 3 bulannya?
- **Channel:** channel dominan per office dan office mana yang polanya berbeda?
- **Distribution:** bandingkan mean, median/p50, p90, p99 dan identifikasi outlier.

**Deliverable:** 1 query utama + 1 `explain()` + 3 insight + 1 keterbatasan.